# Module 7 — SHAP Explainability & Model Interpretability
**PriceMind AI**

This notebook demonstrates model-based SHAP explanations for the production XGBoost demand model.

NOTE: SHAP values reflect learned model input-output relationships. They do NOT imply causality.


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

print("✓ Imports OK")


✓ Imports OK


In [2]:
features_df = pd.read_parquet("../data/processed/features.parquet")
print(f"Features shape: {features_df.shape}")
features_df.head(3)


Features shape: (5055, 88)


,date,sku_id,sku_name,category,store_id,price,cost_price,units_sold,revenue,competitor_price,...,is_excess_stock_flag,log_inventory_level,category_Accessories,category_Hardware & Tools,category_IoT Hardware,category_Software,store_id_STORE-NORTH-01,store_id_STORE-ONLINE-GLOBAL,store_id_STORE-WEST-02,sku_frequency_weight
0,2025-09-29,SKU-1090-CAB,Armored Industrial Bus Cable 50m,Accessories,STORE-NORTH-01,66.55,48.0,43,2861.65,66.67,...,0,5.075174,1,0,0,0,1,0,0,0.2
1,2025-09-30,SKU-1090-CAB,Armored Industrial Bus Cable 50m,Accessories,STORE-NORTH-01,65.67,48.0,53,3480.51,64.47,...,0,5.786897,1,0,0,0,1,0,0,0.2
2,2025-10-01,SKU-1090-CAB,Armored Industrial Bus Cable 50m,Accessories,STORE-NORTH-01,67.89,48.0,51,3462.39,63.59,...,0,6.481577,1,0,0,0,1,0,0,0.2


In [3]:
from ml.explainability.explainer import ShapExplainer

explainer = ShapExplainer()
explainer.load()

print(f"Model: {explainer.model_name}")
print(f"Features: {len(explainer.feature_names)}")
print(f"Base value E[f(X)]: {explainer.base_value:.4f}")


Model: XGBRegressor
Features: 79
Base value E[f(X)]: 18.8872


In [4]:
from ml.explainability.global_explanations import GlobalExplainer

ge = GlobalExplainer(explainer, max_samples=300)
global_df = ge.compute_global_importance(features_df, max_samples=300)

print(f"Global importance computed for {len(global_df)} features")
print("\nTop 15 features by mean |SHAP|:")
print(global_df.head(15)[["rank", "feature", "mean_abs_shap"]].to_string(index=False))


Global importance computed for 79 features

Top 15 features by mean |SHAP|:
 rank                    feature  mean_abs_shap
    1               demand_lag_7       4.596947
    2              demand_lag_28       2.836067
    3              demand_lag_14       2.490256
    4                  log_price       0.970525
    5                day_of_week       0.710929
    6     demand_rolling_mean_28       0.602208
    7       price_to_cost_markup       0.469096
    8                   week_sin       0.328929
    9      demand_rolling_mean_7       0.260418
   10                price_lag_1       0.247582
   11     demand_rolling_mean_14       0.164791
   12 price_deviation_rolling_28       0.144434
   13       demand_rolling_std_7       0.135373
   14                   week_cos       0.122168
   15           competitor_price       0.120099


In [5]:
ge.save_report(global_df, output_path="../reports/shap_global_importance.csv")
print("✓ Saved shap_global_importance.csv")


[GlobalExplainer] Saved global importance → ..\reports\shap_global_importance.csv
✓ Saved shap_global_importance.csv


In [6]:
top20 = ge.top_features(global_df, n=20)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top20["feature"][::-1], top20["mean_abs_shap"][::-1], color="steelblue")
ax.set_xlabel("Mean |SHAP Value| (model contribution magnitude)")
ax.set_title("Top 20 Features — Global SHAP Importance\n(Model input-output relationship, not causal)")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("../reports/shap_global_importance.png", dpi=120, bbox_inches="tight")
plt.show()
print("✓ Saved shap_global_importance.png")


✓ Saved shap_global_importance.png


In [7]:
from ml.explainability.local_explanations import LocalExplainer

le = LocalExplainer(explainer)

# Sample 50 rows for local explanations
sample_100 = features_df.sample(n=50, random_state=42)
X_aligned = explainer.align_features(sample_100)

explanations = le.explain_batch(X_aligned, top_n=10, max_rows=50)
print(f"Generated {len(explanations)} local explanations")
print(f"\nExample (row 0):")
exp0 = explanations[0]
print(f"  Base value      : {exp0.base_value:.4f}")
print(f"  Predicted demand: {exp0.predicted_value:.4f}")
print(f"  Model prediction: {exp0.model_prediction:.4f}")
print(f"  Additive gap    : {abs(exp0.predicted_value - exp0.model_prediction):.6f}")
print(f"  Additive OK?    : {exp0.additive_consistent}")
print(f"  Top positive    : {[(c.feature_name, round(c.shap_value,4)) for c in exp0.top_positive_contributors[:3]]}")
print(f"  Top negative    : {[(c.feature_name, round(c.shap_value,4)) for c in exp0.top_negative_contributors[:3]]}")


Generated 50 local explanations

Example (row 0):
  Base value      : 18.8872
  Predicted demand: 15.1113
  Model prediction: 15.1041
  Additive gap    : 0.007216
  Additive OK?    : True
  Top positive    : [(np.str_('demand_lag_14'), 0.5311), (np.str_('demand_lag_28'), 0.5112), (np.str_('inventory_change_1d'), 0.1548)]
  Top negative    : [(np.str_('day_of_week'), -1.8676), (np.str_('demand_lag_7'), -1.6356), (np.str_('price_to_cost_markup'), -0.3174)]


In [8]:
le.save_report(explanations, output_path="../reports/shap_local_explanations.csv")
print("✓ Saved shap_local_explanations.csv")


[LocalExplainer] Saved 50 local explanations → ..\reports\shap_local_explanations.csv
✓ Saved shap_local_explanations.csv


In [9]:
from ml.explainability.pricing_explanations import PricingExplainer

pe = PricingExplainer(
    explainer,
    features_path="../data/processed/features.parquet",
    elasticity_path="../reports/elasticity_summary.csv",
    recommendations_path="../reports/pricing_recommendations.csv",
)

pricing_exps = pe.explain_all_skus(top_n=10)
print(f"\nPricing explanations generated for {len(pricing_exps)} SKUs")


[PricingExplainer] ✓ SKU-1090-CAB: price SHAP delta = +0.0006, demand Δ = +0.00


[PricingExplainer] ✓ SKU-3320-SENS: price SHAP delta = +0.0000, demand Δ = +0.00
[PricingExplainer] ✓ SKU-4412-MTR: price SHAP delta = +0.0000, demand Δ = +0.00


[PricingExplainer] ✓ SKU-7731-SFT: price SHAP delta = +0.0000, demand Δ = +0.00


[PricingExplainer] ✓ SKU-8921-PRO: price SHAP delta = -0.0945, demand Δ = -0.12

Pricing explanations generated for 5 SKUs


In [10]:
pe.save_report(pricing_exps, output_path="../reports/shap_pricing_explanations.csv")

pricing_df = pe.to_dataframe(pricing_exps)
print("\nPricing Scenario Summary:")
display_cols = ["sku_id", "current_price", "recommended_price",
                "current_predicted_demand", "recommended_predicted_demand",
                "demand_change", "price_shap_delta", "elasticity"]
print(pricing_df[display_cols].to_string(index=False))
print("\n✓ Saved shap_pricing_explanations.csv")


[PricingExplainer] Saved 5 pricing explanations → ..\reports\shap_pricing_explanations.csv

Pricing Scenario Summary:
       sku_id  current_price  recommended_price  current_predicted_demand  recommended_predicted_demand  demand_change  price_shap_delta  elasticity
 SKU-1090-CAB          65.74              74.51                 45.668503                     45.668505       0.000002          0.000633     -0.1827
SKU-3320-SENS         161.87             167.27                 24.406605                     24.406605       0.000000          0.000000     -2.3603
 SKU-4412-MTR         535.03             633.12                 11.035636                     11.035636       0.000000          0.000000     -0.9888
 SKU-7731-SFT         974.90            1121.13                  5.765419                      5.765419       0.000000          0.000000     -0.7109
 SKU-8921-PRO         380.53             450.29                 15.621899                     15.497933      -0.123967         -0.094460 

## Module 7 Summary

| Report | Description |
|--------|-------------|
| shap_global_importance.csv | Mean abs SHAP per feature across sample |
| shap_local_explanations.csv | Per-row top positive/negative SHAP contributors |
| shap_pricing_explanations.csv | Current vs recommended price SHAP delta per SKU |

All explanations are model-based, reflecting the trained XGBoost model learned relationships.
They do NOT imply real-world causal effects.
